# Paper Reproduction: CNN-LSTM-AM for Bearing RUL

This notebook reproduces the core workflow of **Life prediction method of rolling bearing based on CNN-LSTM-AM** inside the project framework: feature extraction, CNN local feature encoding, LSTM temporal modeling, attention pooling, and RUL regression.

In [ ]:
from pathlib import Path

import pandas as pd

from USTC.SSE.BearingPrediction.examples import run_paper_cnn_lstm_attention_reproduction

result = run_paper_cnn_lstm_attention_reproduction(max_samples_per_entity=24, prefer_real_data=True)
comparison_frame = pd.read_csv(result["comparison_path"])
display_columns = [
    "dataset_name",
    "entity_id",
    "model_name",
    "rmse",
    "normalized_rmse",
    "huang_rul_score",
    "rmse_reduction_pct",
    "huang_score_change_pct",
    "over_prediction_rate",
    "within_10_percent_rate",
    "epoch_count",
]
print(comparison_frame[display_columns])
print({"comparison_path": result["comparison_path"], "runs": len(result["runs"])})

assert result["status"] == "OK"
assert result["feature_sequence_shape"][1:] == [5, 19]
assert result["used_dataset_count"] == 2
assert result["trained_model_count"] == 4
assert set(comparison_frame["dataset_name"]) == {"XJTU-SY", "PHM2012"}
assert set(comparison_frame["model_name"]) == {"CNN-LSTM-AM", "CNN-LSTM"}
assert "mae" in result["metrics"]
assert "rmse" in result["metrics"]
assert "huang_rul_score" in result["metrics"]
assert "normalized_rmse" in result["metrics"]
assert "huang_rul_score" in comparison_frame.columns
assert "rmse_reduction_pct" in comparison_frame.columns
assert Path(result["prediction_path"]).exists()
assert Path(result["attention_path"]).exists()

for run in result["runs"]:
    history_frame = pd.read_csv(run["history_path"])
    prediction_frame = pd.read_csv(run["prediction_path"])
    attention_frame = pd.read_csv(run["attention_path"])
    assert len(history_frame) >= 1
    assert len(prediction_frame) == run["prediction_count"]
    if run["use_attention"]:
        assert len(attention_frame) == run["prediction_count"]